# GPU02 — خ۷ (F07) گروه ۷-ب: شبکه‌ی عصبی روی **سطح فرد (L5)** + تجمیع پواسون-دوجمله‌ای

> بند 7.16 (انتظار: 🟢 «جدی‌ترین شانس این خانواده») + بند 7.24.3 (تجمیع از سطح فرد).

**سؤال محوری این نوت‌بوک:** آیا یک مدل کامل سطح فرد — با embedding خودِ `PersonId`
روی ۲٬۰۴۹٬۳۲۲ رزرو و ۲۶٬۷۶۸ نفر — چیزی می‌دهد که فیچرهای تجمیع‌شده ندادند؟

⚠️ اسپرینت B همین فرضیه را از مسیر دیگری **رد کرد** (یافته‌ی ۲۱: فیچرهای کوهورت
L5→L1 کمک نکردند چون سیگنال فردی از قبل در فیچرهای rolling سلولی حاضر بود). این
نوت‌بوک همان فرضیه را در **قوی‌ترین شکل ممکنش** می‌آزماید. اگر این هم نبرد، رد
فرضیه دیگر «شاید تجمیع بد بود» ندارد — و این خودش یک نتیجه‌ی قابل‌گزارش است.

### نکته‌ی فنی محوری: تجمیع (بند 7.24.3)

مدل به‌ازای هر رزرو $p_i$ می‌دهد؛ تعداد عدم‌دریافت سلول = مجموع برنولی‌های
ناهم‌توزیع ⇒ **پواسون-دوجمله‌ای**، و کوانتایلش با بسط کورنیش-فیشر گرفته می‌شود.
ولی پواسون-دوجمله‌ای **استقلال افراد** را فرض می‌کند در حالی که F59 می‌گوید ۸۳٪
واریانس شوک مشترک روزانه است. به همین دلیل یک هایپرپارامتر صریح `overdispersion`
هست که Optuna تنظیمش می‌کند — **مقدار بهینه‌اش خودش اندازه‌گیری همبستگی درون‌روزی
است**، نه یک وصله.

**ارزیابی روی همان ردیف‌های L1** انجام می‌شود که همه‌ی خانواده‌های دیگر با آن سنجیده
شدند (بند 7.1.2)، وگرنه عدد pinball با هیچ ردیف دیگری قابل‌قیاس نبود.

**بودجه‌ی هدف: ~۱۰۰ دقیقه.** سنگین‌ترین نوت‌بوک از این چهارتا.

## سلول ۱ — نصب وابستگی‌ها

`torch`/`jax` روی کولب و کگل از پیش نصب‌اند و نسخه‌شان با درایور CUDA همان ماشین
هماهنگ است؛ نصب دوباره‌شان چند گیگابایت دانلود و گاهی ناسازگاری درایور می‌آورد.
پس فقط چیزهایی نصب می‌شوند که واقعاً نیستند. نسخه‌ی دقیق هرچه استفاده شد در سلول ۵
چاپ و در MLflow ثبت می‌شود (بازتولیدپذیری از راه **ثبت**، نه پین‌کردن).
فهرست کامل: `requirements-gpu.txt` داخل همین بسته.

In [ ]:
!pip install -q optuna mlflow

## سلول ۲ — بارگذاری بسته‌ی کد + داده

⚠️ **کد اصلی داخل نوت‌بوک نوشته نمی‌شود** (بند 7.8.4، قاعده‌ی «`notebooks/` = روایت،
`src/` = حقیقت»). این نوت‌بوک فقط `src/` را import و روایت می‌کند.

`gpu_bundle.zip` را با `python -m src.models.gpu_bundle` بسازید و در Drive بگذارید
(یا در کگل به‌عنوان Dataset آپلود کنید). داخلش: کل `src/`، چهار فایل
`data/processed/` که سلول ۳ رویشان assert می‌زند، و نتایج CPU خانواده‌های قبلی برای
جدول مقایسه.

In [ ]:
MODE = "colab"          # ← "colab" یا "kaggle"
BUNDLE_COLAB  = "/content/drive/MyDrive/phase7/gpu_bundle.zip"   # ← مسیر خودتان
BUNDLE_KAGGLE = "/kaggle/input/phase7-bundle/gpu_bundle.zip"

import os, sys, zipfile, pathlib

if MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    bundle, workdir = BUNDLE_COLAB, pathlib.Path("/content/phase7")
else:
    bundle, workdir = BUNDLE_KAGGLE, pathlib.Path("/kaggle/working/phase7")

workdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall(workdir)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)

**اگر هش‌ها نخوانند، نوت‌بوک همین‌جا می‌ایستد.** بدون این assert هیچ اثباتی وجود
ندارد که این اجرا روی همان foldها و همان snapshot دادهٔ خانواده‌های CPU انجام شده —
و هر run با `cv_folds_hash` نامنطبق از جدول مقایسه‌ی فاز ۷ حذف می‌شود. مقادیر زیر
از بخش «قفل فاز ۷» `doc/data_manifest.md` آمده‌اند.

In [ ]:
from src.models.gpu_runner import assert_fairness_gate

EXPECTED_CV_FOLDS_HASH      = "bd08d6f7c801ee0611121e404774251de07a480ac1589b12eb7c64f8044b78d4"
EXPECTED_DATA_SNAPSHOT_HASH = "90097e5f3b7d4572ee94c9a1a09ae0d0ec65115c413445a4751e20c48ac35de0"   # data/processed/person_features_v1.parquet

from src.models.families.f07_neural_l5 import load_l5_bridge
data = load_l5_bridge()   # ارزیابی L1، آموزش L5 — بند بالای ماژول

assert_fairness_gate(data, EXPECTED_CV_FOLDS_HASH, EXPECTED_DATA_SNAPSHOT_HASH)
print(data.summary())

## سلول ۴ — بذر تصادفی سراسری

`set_global_seed()` تنها منبع بذر پروژه است (`AGENTS.md`). قطعیت کامل روی GPU
تضمین‌شدنی نیست — به‌همین‌دلیل قاعده‌ی **سه seed** (A7، بند 7.16.3) در مرحله‌ی
قهرمان اجرا می‌شود و پراکندگی بین seedها خودش گزارش می‌گردد، نه پنهان.

In [ ]:
from src.config import set_global_seed
from src.models.gpu_runner import setup_torch_determinism

set_global_seed()
setup_torch_determinism(strict=False)   # strict=True بعضی op های cuDNN را می‌شکند

## سلول ۵ — سخت‌افزار

زمان‌های اجرا فقط با دانستن سخت‌افزار قابل تفسیرند (بند 7.8.2).

In [ ]:
!nvidia-smi

from src.models.gpu_runner import device_report

DEVICE = device_report()
DEVICE

## سلول ۶ — ردیابی MLflow جدا

`mlruns_gpu/` جداست تا ادغام با `mlruns/` محلی (بند 7.8.3 گام ۵) امن و قابل بازگشت
باشد. tag اجباری `compute` هم همین‌جا ست می‌شود.

In [ ]:
from pathlib import Path
from src.models.gpu_runner import use_gpu_tracking

COMPUTE = "colab"     # اگر روی کگل اجرا می‌کنید: "kaggle"
print("MLflow →", use_gpu_tracking("mlruns_gpu"))

# reports/gpu/ را همین‌جا می‌سازیم — سلول‌های بعدی مستقیم CSV آن‌جا می‌نویسند،
# پیش از آنکه save_family_report/package_outputs بسازدش
Path("reports/gpu").mkdir(parents=True, exist_ok=True)


## سلول ۷-الف — R0: آزمایش دود روی ۲ میلیون رزرو

با هایپرپارامتر پیش‌فرض و epoch کم — فقط برای اثبات اینکه پایپ‌لاین کامل
(برازش سطح فرد ← احتمال هر رزرو ← تجمیع پواسون-دوجمله‌ای ← merge روی سلول‌های L1)
سرتاسر کار می‌کند و عدد معنادار می‌دهد.

In [ ]:
from src.models.families import f07_neural_l5 as fam
from src.models.gpu_runner import smoke_test

person = fam.load_person_frame()
print(f"جدول رزرو فردی: {len(person):,} ردیف · {person['PersonId'].nunique():,} فرد · "
      f"نرخ عدم‌دریافت کل = {person['dont_receive'].mean():.4f}")

smoke = [smoke_test(fam.FITTERS["mlp_embedding_l5"], data,
                    hyperparams={"epochs": 3, "patience": 2})]

## سلول ۷-ب — R2: تنظیم روی ۳ fold نخست

هر برازش این‌جا گران است (میلیون‌ها ردیف)، پس منطق R1 بند 7.3.2 اعمال می‌شود:
**تنظیم روی ۳ fold نخست، تأیید نهایی روی هر ۵**. این دقیقاً همان صرفه‌جویی است که
سند تصمیم ۳۷ خواسته — بدون آن، هر trial پنج برازش کامل می‌شد.

In [ ]:
from src.models.gpu_runner import run_gpu_study
from src.models.spaces import SPACES

study = run_gpu_study(
    fam.FITTERS["mlp_embedding_l5"], SPACES["mlp_embedding_l5"].fn,
    data.first_folds(3),                       # ← غربالگری ارزان، بند 7.3.2
    family=fam.FAMILY, feature_set=fam.FEATURE_SET,
    budget_minutes=55, compute=COMPUTE, seed=42,
    target="binary", output_aggregation="aggregated")   # ← محورها صریح، بند 7.9.1

## سلول ۷-ج — قهرمان روی هر ۵ fold + ACI + DM

دو seed (نه سه): هر seed این‌جا یعنی ۵ برازش کامل روی میلیون‌ها ردیف. پراکندگی
بین همین دو هم اگر بزرگ باشد، کافی است که «برد» را زیر سؤال ببرد.

In [ ]:
from src.models.gpu_runner import finalize_champion

champion = finalize_champion(fam.FITTERS["mlp_embedding_l5"], data, study,
                             feature_set=fam.FEATURE_SET, seeds=(42, 1234),
                             compute=COMPUTE, run_aci=True,
                             target="binary", output_aggregation="aggregated")
champions = [champion]

## سلول ۷-د — ⭐ آزمون تجمیع (بند 7.24.3): کوانتایل پواسون-دوجمله‌ای در برابر میانگین

سه حالت روی همان مدل و همان احتمال‌ها مقایسه می‌شوند:

| حالت | معنی |
|---|---|
| `mean_only` | فقط میانگین $\sum p_i / n$ — یعنی هیچ کوانتایلی گرفته نشده |
| `cornish_fisher`, `overdispersion=1` | کوانتایل با فرض **استقلال کامل** افراد |
| `cornish_fisher`, `overdispersion` تنظیم‌شده | با احتساب همبستگی درون‌روزی |

فاصله‌ی ردیف دوم تا سوم، **قیمت فرض استقلال** است — همان چیزی که بند 7.24.3 هشدار
می‌دهد و تا امروز در این پروژه هرگز کمّی نشده بود.

In [ ]:
import numpy as np, pandas as pd
from src.baselines import operational_metrics
from src.models.axes import TUNING_TAU

hp = dict(study.best_hyperparams)
tr0, te0 = data.folds[0]
model0 = fam.FITTERS["mlp_embedding_l5"].fit(tr0, TUNING_TAU, **hp)

rows = []
for label, mode, od in [("میانگین (بدون کوانتایل)", "mean_only", 1.0),
                        ("پواسون-دوجمله‌ای، استقلال کامل", "cornish_fisher", 1.0),
                        (f"پواسون-دوجمله‌ای، overdispersion={hp.get('overdispersion', 1.0):.2f}",
                         "cornish_fisher", float(hp.get("overdispersion", 1.0)))]:
    model0.arch["aggregation_mode"], model0.arch["overdispersion"] = mode, od
    pred = fam.predict_l5(model0, te0, TUNING_TAU)
    m = operational_metrics(te0, pred, TUNING_TAU)
    rows.append({"حالت تجمیع": label, "pinball": round(m["pinball"], 5),
                 "پوشش": round(m["coverage"], 4), "شکاف از τ": round(m["coverage_gap"], 4),
                 "نرخ کمبود": round(m["shortage_rate"], 4)})

aggregation_table = pd.DataFrame(rows)
aggregation_table.to_csv("reports/gpu/F07b_aggregation_ablation.csv", index=False)
aggregation_table

## سلول ۷-ه — راستی‌آزمایی مدل ذخیره‌شده

In [ ]:
from pathlib import Path

stem = Path("models/gpu/F07/mlp_embedding_l5/mlp_embedding_l5__s42__fold0")
reloaded = fam.FITTERS["mlp_embedding_l5"].load(stem)
pred_reloaded = fam.predict_l5(reloaded, data.folds[0][1], TUNING_TAU)
print(f"پارامترها: {reloaded.n_parameters:,} · ردیف‌های آموزش سطح فرد: "
      f"{reloaded.arch['n_person_rows_train']:,}")
assert np.isfinite(pred_reloaded).all()
print("✅ مدل ذخیره‌شده قابل استفاده است")

## سلول ۷-و — گزارش فارسی کامل

In [ ]:
from src.models.gpu_runner import render_family_report, save_family_report

notes = [
    "مدل روی L5 آموزش دید ولی ارزیابی روی همان ردیف‌های L1 انجام شد (بند 7.1.2) — "
    "چون داده‌ی فردی FoodType ندارد، پیش‌بینی هر سلول (d,m,r) روی همه‌ی سطرهای هم‌غذا پخش شد.",
    f"overdispersion بهینه = {study.best_hyperparams.get('overdispersion', float('nan')):.3f} — "
    "بزرگ‌تر از ۱ یعنی فرض استقلال افراد (بند 7.24.3) واقعاً نقض می‌شود، سازگار با F59.",
    "این نوت‌بوک فرضیه‌ی یافته‌ی ۲۱ (سیگنال فردی افزونه است) را در قوی‌ترین شکلش می‌آزماید.",
    "تقسیم fold بر اساس تاریخ است نه فرد (بند 7.16.3) — همان فرد می‌تواند در train و test باشد.",
    "⚠️ ستون B3 جدول R2 روی ۳ fold نخست است (تنظیم آن‌جا انجام شد) ولی جدول قهرمان روی هر ۵ fold — "
    "این دو عدد مستقیماً قابل‌قیاس نیستند و عمداً جدا گزارش شده‌اند.",
]
report = render_family_report(
    "F07", "خ۷ گروه ۷-ب — شبکه‌ی عصبی سطح فرد (L5) با تجمیع پواسون-دوجمله‌ای",
    [study], champions, smoke, DEVICE, notes)
save_family_report("F07", report, "F07b_neural_L5")
print(report)

## سلول ۸ — بسته‌بندی خروجی (تکه‌های ۱۰۰ مگابایتی)

همه‌ی خروجی‌ها — `mlruns_gpu/` (هر trial + قهرمان‌ها با artifact مدل)،
`models/gpu/` (وزن‌ها و پیش‌پردازش هر fold/seed)، `reports/gpu/` (JSON و گزارش
فارسی)، و `optuna_studies/*.db` (تا اجرای بعدی از همین‌جا ادامه دهد) — در یک zip
جمع و به تکه‌های ۱۰۰ مگابایتی شکسته می‌شوند. هر تکه SHA-256 خودش را در
`MANIFEST_F07b_neural_L5.json` دارد، پس اگر دانلود یکی خراب شد فقط همان یکی دوباره گرفته
می‌شود.

In [ ]:
from src.models.gpu_runner import package_outputs, download_parts

manifest = package_outputs(tag="F07b_neural_L5", part_mb=100)
download_parts()      # روی کولب دانلود می‌کند؛ روی کگل فایل‌ها در خروجی session می‌مانند

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb    ← حتی اگر آزمایش شکست خورد؛ شکست هم داده است
۲. تکه‌ها → ریشه‌ی مخزن:  cat gpu_outputs_*.zip.part* > gpu_outputs.zip && unzip gpu_outputs.zip
۳. rsync -a mlruns_gpu/ mlruns/        (ادغام MLflow)
۴. کارت مدل ۱۴ گامی → reports/models/{model_id}.md
۵. make mlflow-ui  →  runهای جدید با tag compute=colab باید دیده شوند
```